# 00 — Référence PyTorch eager (CPU et GPU)

Ce notebook n'est pas l'un des quatre pipelines demandés : il établit la
**référence non optimisée**. Toutes les comparaisons de débit, de latence,
d'accuracy et d'accord de prédictions se font par rapport à cette exécution.

- Modèle : ResNet-18 ImageNet + tête linéaire 10 classes (linear probe CIFAR-10).
- Données : 50 images CIFAR-10 de test (5 par classe), redimensionnées en 224×224
  et normalisées avec les paramètres ImageNet.
- Protocole : 5 répétitions indépendantes, 5 warmups, batch de 10, 50 images.
- Mesures : débit (images/s), latence (ms/image), énergie CPU/GPU, Top-1.

In [ ]:
%matplotlib inline
import json
import os
import sys
import time
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "src" / "common.py").exists():
            return candidate
    return p

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TP_ROOT", str(ROOT))
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("TP_REPO_URL", "")
if IN_COLAB and REPO_URL and not (ROOT / "src" / "common.py").exists():
    import subprocess
    subprocess.run(["git", "clone", REPO_URL], check=True)
    ROOT = Path(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")).resolve()
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    os.environ["TP_ROOT"] = str(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from src import common as C

C.ensure_dirs()
C.set_seed()
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "4")))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Racine du projet :", ROOT)
print("PyTorch :", torch.__version__, "| CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print(f"Protocole : {C.N_IMAGES} images | batch={C.BATCH_SIZE} | {C.REPETITIONS} répétitions | warmup={C.WARMUP}")

In [ ]:
def show(result_or_id):
    result = C.load_result(result_or_id) if isinstance(result_or_id, str) else result_or_id
    print(f"{result['label']}  |  device={result['device']}  |  backend={result['backend']}")
    display(pd.DataFrame(C.summary_rows(result)).round(4))
    print(f"Top-1 accuracy = {result['accuracy_top1'] * 100:.2f} %")

def bar_metric(ids, metric, std_metric, ylabel, title, color="#4C72B0"):
    df = pd.DataFrame(C.comparison_rows(ids))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["pipeline"], df[metric], yerr=df[std_metric], capsize=5, color=color)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=18)
    for i, value in enumerate(df[metric]):
        if value is not None:
            ax.text(i, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    return df

## 1. Données et prétraitement

In [ ]:
images, labels = C.load_subset()
x_cpu = C.preprocess_images(images, device="cpu")
print("images :", images.shape, images.dtype)
print("tenseur prétraité :", tuple(x_cpu.shape), x_cpu.dtype)
print("distribution :", np.bincount(labels, minlength=len(C.CLASSES)).tolist())
print("classes :", C.CLASSES)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    ax.imshow(images[idx])
    ax.set_title(C.CLASSES[int(labels[idx])])
    ax.axis("off")
fig.suptitle("Exemples du sous-ensemble CIFAR-10 (5 premières classes)")
plt.tight_layout()
plt.show()

## 2. Modèle commun

In [ ]:
model = C.build_model("cpu")
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 + tête 10 classes : {n_params/1e6:.2f} M paramètres")
print("Couche de classification :", model.fc)
print("Poids de la tête :", C.ARTIFACTS / "classifier_head.npz")

## 3. Référence CPU — PyTorch eager

C'est le pipeline « haut niveau → matériel » sans format intermédiaire ni
runtime optimisé : PyTorch exécute directement les opérateurs sur le CPU.

In [ ]:
ref_cpu = C.ensure_reference("cpu", x_cpu, labels)
show(ref_cpu)

In [ ]:
records = pd.DataFrame(ref_cpu["records"])
ax = records.plot(x="repetition", y="throughput_ips", marker="o", legend=False, figsize=(8, 3))
ax.set_xlabel("Répétition")
ax.set_ylabel("images/s")
ax.set_title("Stabilité du débit — PyTorch eager CPU")
ax.set_xticks(records["repetition"])
plt.tight_layout()
plt.show()
records[["repetition", "throughput_ips", "latency_ms_per_image", "energy_cpu_j"]].round(4)

## 4. Référence GPU — PyTorch eager (si GPU disponible)

Sur Colab, sélectionnez un runtime GPU. En local, le conteneur est lancé avec
`--gpus all`.

In [ ]:
if torch.cuda.is_available():
    ref_gpu = C.ensure_reference("cuda", x_cpu, labels)
    show(ref_gpu)
else:
    ref_gpu = None
    print("GPU indisponible : la référence GPU ne peut pas être calculée.")

## 5. Prédictions de la référence CPU

In [ ]:
logits = np.load(C.RESULTS / "00_reference_pytorch_cpu_logits.npy")
pred = logits.argmax(axis=1)
df_pred = pd.DataFrame({
    "image": np.arange(len(labels)),
    "vraie classe": [C.CLASSES[int(i)] for i in labels],
    "prédiction": [C.CLASSES[int(i)] for i in pred],
    "correct": pred == labels,
    "confiance": logits.max(axis=1),
})
display(df_pred.head(20).round(3))
print(f"Top-1 sur les 50 images : {C.top1_accuracy(logits, labels) * 100:.2f} %")

## 6. Comparaison CPU / GPU de la référence

Les deux exécutions utilisent exactement les mêmes poids et les mêmes entrées.
L'écart de débit attendu est important : le GPU est conçu pour les opérations
matricielles massives, tandis que le CPU reste limité par ses cœurs et sa
hiérarchie mémoire.

In [ ]:
ids = ["00_reference_pytorch_cpu"]
if ref_gpu is not None:
    ids.append("00_reference_pytorch_gpu")
table = pd.DataFrame(C.comparison_rows(ids))
display(table.round(4))
bar_metric(ids, "throughput_ips", "throughput_std", "images/s",
           "Référence PyTorch eager : débit CPU vs GPU")

### Interprétation

La référence mesure le coût des opérateurs PyTorch sans optimisation de
déploiement. Les notebooks suivants vont comparer chaque pipeline optimisé à
cette base. La variabilité (écart-type) est calculée sur les 5 répétitions :
elle permet de distinguer un vrai gain d'un simple bruit de mesure.